# Day 3: Probability & Information Theory

**Module 1 — Foundations | 100 Days of Data Science**

## Why This Matters
Every classifier outputs **probabilities**, and every loss function that trains it is built from **information theory**. Cross-entropy loss — the single most-used loss function in deep learning — comes directly from these ideas.

By the end of today you'll understand not just *how* to call `nn.CrossEntropyLoss()`, but *why* it's shaped the way it is.

## Topics Covered Today
1. Probability basics (distributions, expectation)
2. Softmax — turning scores into probabilities
3. Entropy — measuring uncertainty
4. Cross-entropy — measuring the gap between two distributions
5. KL divergence
6. Cross-entropy loss from scratch (mini classifier example)
7. Practice exercises

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("NumPy version:", np.__version__)

---
## 1. Probability Basics

A **probability distribution** assigns a likelihood to each possible outcome. For classification, the model outputs a distribution over classes — e.g. `[0.7, 0.2, 0.1]` for `[cat, dog, bird]`.

Two rules always hold:
- Every probability is between 0 and 1
- All probabilities in a distribution sum to 1

**Expectation** (expected value) is the weighted average outcome:
$$E[X] = \sum_i p_i \cdot x_i$$

In [ ]:
outcomes = np.array([1, 2, 3, 4, 5, 6])  # a die
probs = np.array([1/6]*6)

expected_value = np.sum(outcomes * probs)
print("Expected value of a fair die roll:", expected_value)

# A model's predicted class distribution
class_probs = np.array([0.7, 0.2, 0.1])
print("Sum of probabilities (should be 1):", class_probs.sum())

---
## 2. Softmax — Turning Raw Scores Into Probabilities

A neural network's final layer outputs raw scores ("logits") — not probabilities. **Softmax** converts them into a valid probability distribution.

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

In [ ]:
def softmax(logits):
    exp_scores = np.exp(logits - np.max(logits))  # subtract max for numerical stability
    return exp_scores / np.sum(exp_scores)

logits = np.array([2.0, 1.0, 0.1])  # raw model output for 3 classes
probs = softmax(logits)

print("Logits:", logits)
print("Softmax probabilities:", probs)
print("Sum:", probs.sum())

plt.bar(['cat', 'dog', 'bird'], probs, color=['#4C72B0', '#DD8452', '#55A868'])
plt.title('Softmax Output Distribution')
plt.ylabel('Probability')
plt.show()

---
## 3. Entropy — Measuring Uncertainty

Entropy quantifies how "uncertain" or "spread out" a distribution is.

$$H(p) = -\sum_i p_i \log(p_i)$$

- Low entropy = distribution is confident/concentrated (e.g. `[0.99, 0.01]`)
- High entropy = distribution is uncertain/spread out (e.g. `[0.5, 0.5]`)

In [ ]:
def entropy(p):
    p = np.clip(p, 1e-12, 1)  # avoid log(0)
    return -np.sum(p * np.log(p))

confident_dist = np.array([0.99, 0.01])
uncertain_dist = np.array([0.5, 0.5])

print("Entropy of confident distribution [0.99, 0.01]:", entropy(confident_dist))
print("Entropy of uncertain distribution [0.5, 0.5]:", entropy(uncertain_dist))

In [ ]:
# Visualize entropy across all possible probabilities for a binary outcome
p_values = np.linspace(0.001, 0.999, 200)
entropies = [-(p*np.log(p) + (1-p)*np.log(1-p)) for p in p_values]

plt.figure(figsize=(6,4))
plt.plot(p_values, entropies)
plt.xlabel('P(class = 1)')
plt.ylabel('Entropy')
plt.title('Entropy is Maximized at Maximum Uncertainty (p=0.5)')
plt.grid(True)
plt.show()

---
## 4. Cross-Entropy — Comparing Two Distributions

Cross-entropy measures the difference between the **true distribution** (the actual label) and the **predicted distribution** (the model's output). This is the loss function used in essentially every classification model.

$$H(p, q) = -\sum_i p_i \log(q_i)$$

where `p` = true labels, `q` = predicted probabilities.

For a single training example with true class `y`, this simplifies to just:
$$\text{Loss} = -\log(q_y)$$
(i.e. the negative log of the probability the model assigned to the correct class).

In [ ]:
def cross_entropy(true_dist, pred_dist):
    pred_dist = np.clip(pred_dist, 1e-12, 1)
    return -np.sum(true_dist * np.log(pred_dist))

# True label: class 0 (one-hot encoded)
true_label = np.array([1, 0, 0])

# Two different model predictions
good_prediction = np.array([0.9, 0.08, 0.02])
bad_prediction = np.array([0.2, 0.5, 0.3])

print("Cross-entropy loss (confident & correct):", cross_entropy(true_label, good_prediction))
print("Cross-entropy loss (uncertain & wrong):", cross_entropy(true_label, bad_prediction))
print("\n-> Notice: the worse the prediction, the higher the loss.")

---
## 5. KL Divergence

KL (Kullback-Leibler) divergence measures how much one distribution differs from another. It's related to cross-entropy by:

$$D_{KL}(p \| q) = H(p, q) - H(p)$$

In other words: **cross-entropy = entropy + KL divergence**. Since the true label's entropy `H(p)` is usually 0 (one-hot labels are fully certain), minimizing cross-entropy is effectively the same as minimizing KL divergence — pushing the model's distribution as close as possible to the true one.

In [ ]:
def kl_divergence(p, q):
    p = np.clip(p, 1e-12, 1)
    q = np.clip(q, 1e-12, 1)
    return np.sum(p * np.log(p / q))

print("KL divergence (good prediction):", kl_divergence(true_label, good_prediction))
print("KL divergence (bad prediction):", kl_divergence(true_label, bad_prediction))
print("\nCompare to cross-entropy values above -- since H(p)=0 for one-hot labels, they match cross-entropy exactly.")

---
## 6. Full Mini Classifier Example
Putting it together: raw logits → softmax → cross-entropy loss, exactly like the final layer of a real classifier.

In [ ]:
# Simulate a 3-class classifier for a single example
true_class_index = 1  # true label is class "1"
num_classes = 3
true_one_hot = np.eye(num_classes)[true_class_index]

logits = np.array([1.2, 3.1, 0.4])  # raw output from the last layer

probs = softmax(logits)
loss = cross_entropy(true_one_hot, probs)

print("True label (one-hot):", true_one_hot)
print("Predicted probabilities:", np.round(probs, 4))
print("Cross-entropy loss:", loss)

try:
    import torch
    import torch.nn as nn

    logits_t = torch.tensor([logits])
    target_t = torch.tensor([true_class_index])
    criterion = nn.CrossEntropyLoss()
    torch_loss = criterion(logits_t, target_t)
    print("\nPyTorch CrossEntropyLoss (should match):", torch_loss.item())
except ImportError:
    print("PyTorch not installed. Run: pip install torch")

---
## 7. Practice Exercises
Try these before Day 4:

1. Compute softmax for logits `[1.0, 2.0, 3.0, 4.0]` and verify the probabilities sum to 1.
2. Compute entropy for the distribution `[0.25, 0.25, 0.25, 0.25]` — is it higher or lower than `[0.7, 0.1, 0.1, 0.1]`? Why?
3. Given true label = class 2 (out of 4 classes), compute cross-entropy loss for two different predicted distributions and compare.
4. Compute KL divergence between `[0.5, 0.5]` and `[0.9, 0.1]` in both directions — is `KL(p||q) == KL(q||p)`?
5. In your own words: why does cross-entropy loss punish confident-and-wrong predictions much more than uncertain-and-wrong ones?

In [ ]:
# Your practice code here


---
## Summary
- **Softmax** converts raw scores into a valid probability distribution
- **Entropy** measures how uncertain a distribution is
- **Cross-entropy** measures the gap between predicted and true distributions — this IS the loss function behind most classifiers
- **KL divergence** measures how one distribution diverges from another; minimizing cross-entropy ≈ minimizing KL divergence for one-hot labels
- These three ideas together explain exactly why `CrossEntropyLoss` looks the way it does

**Module 1 — Foundations is now complete! (Day 1: Linear Algebra, Day 2: Calculus, Day 3: Probability & Info Theory)**

Next up: **Day 4 — Start of Module 2: Neural Network Basics (Perceptron → MLP)**

---
*Part of the 100 Days of Data Science series | DL-for-Data-Science repo*